<a href="https://colab.research.google.com/github/jeyner99/Integracion-de-datos-y-prospectiva/blob/main/Reto_3_integracion_de_datos_y_prospectiva_Jeyner_Casta%C3%B1o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de estudio US Health Insurance

Una aseguradora de salud con cobertura nacional en los Estados Unidos
quiere mejorar la cobertura preventiva de sus afiliados. La entidad necesita
identificar patrones de riesgo epidemiológico para reorganizar la oferta de
servicios médicos e incentivos de salud, en lugar de trasladar a los
pacientes, para lo cual quiere crear unos clusters en salud en donde
quiere analizar la distribución territorial de estos grupos por región

 La entidad quiere identificar el número de clusters que pueden ser
eficientes para la reorganización de la oferta de servicios médicos.

✓ Quiere caracterizar las variables clínicas y los perfiles de los
pacientes de acuerdo con sus variables epidemiológicas (Variables
de Entrada) y de costo siniestro (Variable de Salida).

✓ La entidad quiere conocer la distribución porcentual por región y por
siniestralidad (Variables Salida)..

##Cargamos las librerías y los datos desde Kaggle

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Download latest version
path = kagglehub.dataset_download("teertha/ushealthinsurancedataset")

# List files in the dataset directory
file_list = os.listdir(path)
print(f"Files in the dataset directory: {file_list}")

# Assuming the main dataset file is 'insurance.csv'
dataset_file = os.path.join(path, 'insurance.csv')

# Read the dataset into a pandas DataFrame
XDB = pd.read_csv(dataset_file)


XDB2 = XDB.iloc[:,[0,1,2,3,4,5,6]].copy() #Están todas las variables


XDB = XDB.iloc[:, [0,1,2,3,4,5]] #Solo están las variables de entrada

100%|██████████| 16.0k/16.0k [00:00<00:00, 21.9MB/s]

Extracting files...
Files in the dataset directory: ['insurance.csv']


##Transformamos las variables 'sex' y 'smoker' en categoricas para que puedan aportar a la clasificación del paciente.

In [ ]:
for k in range(len(XDB)):

  if XDB.iloc[k,1] == 'female': #sex: 1 si es mujer, 0 si es hombre
    XDB.iloc[k,1] = 1
  else:
    XDB.iloc[k,1] = 0

  if XDB.iloc[k,4] == 'yes': #smoker: 1 si fuma, 0 si no
    XDB.iloc[k,4] = 1
  else:
    XDB.iloc[k,4] = 0

##En este paso lo que hacemos es visualizar el promedio de las variables para cada región, está información es útil para un analisis exploratorio pero lo que nosotros haremos será ir más a fondo clasificando a las pacientes según si tienen caracterisiticas similares.

In [ ]:
Xciu = XDB2['region'].unique()

Xciu = ['southwest', 'southeast', 'northwest', 'northeast'] #Estos serán los concentradores de información

# Identify numeric columns for calculating means
numeric_cols = ['age', 'sex', 'bmi', 'children', 'smoker'] # These are the numerical input variables from XDB

#Se procede con la creación de los clusters

XCm = np.zeros((len(Xciu), len(numeric_cols))) # Adjust size to match numeric columns

#Fase 0: Reconocer los clusters o valores por ciudad
for i, region_name in enumerate(Xciu):
  print(i, region_name)
  filas = np.where(XDB2['region']== region_name)[0]
  # Calculate mean only for numeric columns
  XCm[i,:] = np.mean(XDB.loc[filas, numeric_cols], axis = 0)

df = pd.DataFrame(XCm, columns = numeric_cols) # Use correct column names for the new DataFrame
df.index = Xciu
display(df)

0 southwest
1 southeast
2 northwest
3 northeast


,age,sex,bmi,children,smoker
southwest,39.455385,0.498462,30.596615,1.141538,0.178462
southeast,38.939560,0.480769,33.355989,1.049451,0.250000
northwest,39.196923,0.504615,29.199785,1.147692,0.178462
northeast,39.268519,0.496914,29.173503,1.046296,0.206790


Se procede con la creación de los clusters con **K-Medoids** (k = 4)

Se normalizan las variables de entrada para que todas pesen igual en la distancia (age y bmi tienen escalas mucho mayores que sex o smoker).

In [ ]:
XDB[numeric_cols] = XDB[numeric_cols].astype(float)

scaler = MinMaxScaler()
XN = scaler.fit_transform(XDB[numeric_cols]) #Pacientes normalizados
XCmN = scaler.transform(df) #Promedios por región normalizados

Fase 1: Medoides iniciales

En K-Medoids el centro de cada cluster debe ser un **paciente real**. Por eso, como punto de partida, se toma el paciente más cercano al promedio de cada región.

In [ ]:
k = 4
med = np.zeros(k, dtype = int) #Aquí se guarda la fila del paciente que es medoide

for i in range(k):
  d = np.sum((XN - XCmN[i,:])**2, axis = 1) #Distancia de todos los pacientes al promedio de la región i
  med[i] = np.argmin(d)

print(med)

[ 922  325  669 1198]


Fase 2: Algoritmo K-Medoids

1. Cada paciente se asigna al medoide más cercano.
2. En cada cluster, el nuevo medoide es el paciente con menor distancia total a los demás pacientes del grupo.
3. Se repite hasta que los medoides no cambien.

In [ ]:
for it in range(20):

  #Paso 1: asignar cada paciente al medoide más cercano
  etq = np.zeros(len(XN), dtype = int)
  for i in range(len(XN)):
    d = np.sum((XN[med,:] - XN[i,:])**2, axis = 1)
    etq[i] = np.argmin(d)

  #Paso 2: buscar el nuevo medoide de cada cluster
  med2 = med.copy()
  for j in range(k):
    filas = np.where(etq == j)[0]
    dist = np.sqrt(((XN[filas][:, None, :] - XN[filas][None, :, :])**2).sum(axis = 2))
    med2[j] = filas[np.argmin(dist.sum(axis = 1))]

  print('Iteración', it, med2)

  #Paso 3: si los medoides no cambian, el algoritmo terminó
  if np.all(med2 == med):
    break
  med = med2

Xcl = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3'] #Estos serán los concentradores de información

XCm = XN[med, :] #XCm: los medoides normalizados

dfm = XDB2.iloc[med, :].copy() #Los medoides son pacientes reales
dfm.index = Xcl
display(dfm)

Iteración 0 [1108  630  669  273]


,age,sex,bmi,children,smoker,region,charges,Cluster,Siniestralidad
Cluster 0,26,male,30.000,1,no,southwest,2904.08800,Cluster 0,Baja
Cluster 1,53,male,36.100,1,no,southwest,10085.84600,Cluster 1,Media
Cluster 2,40,female,29.810,1,no,southeast,6500.23590,Cluster 2,Media
Cluster 3,50,male,27.455,1,no,northeast,9617.66245,Cluster 3,Media


Se asigna cada paciente a su cluster

In [ ]:
XCl3 = [] #Aquí voy a almacenar en qué cluster quedó cada paciente

for k2 in range(len(XDB2)):
  d = np.sum((XCm - XN[k2,:])**2, axis = 1) #Distancia de cada paciente a cada medoide
  nc = np.argmin(d)
  XCl3.append(Xcl[nc])

XDB2['Cluster'] = XCl3
XDB2['Cluster'].value_counts()

,count
Cluster,
Cluster 2,662
Cluster 0,325
Cluster 3,192
Cluster 1,159


Caracterización de los clusters (variables clínicas y costo siniestro)

In [ ]:
XCm2 = np.zeros((len(Xcl), 6)) #Variables de entrada + charges

for i, cl in enumerate(Xcl):
  print(i, cl)
  filas = np.where(XDB2['Cluster'] == cl)[0]
  XCm2[i,:5] = np.mean(XDB.loc[filas, numeric_cols], axis = 0)
  XCm2[i,5] = np.mean(XDB2.iloc[filas, 6])

df2 = pd.DataFrame(XCm2, columns = numeric_cols + ['charges'])
df2.index = Xcl
display(df2)

0 Cluster 0
1 Cluster 1
2 Cluster 2
3 Cluster 3


,age,sex,bmi,children,smoker,charges
Cluster 0,26.206154,0.0,30.532292,1.076923,0.243077,10854.199611
Cluster 1,53.144654,0.0,36.638805,1.075472,0.226415,19289.336738
Cluster 2,39.503021,1.0,30.377749,1.074018,0.173716,12569.578844
Cluster 3,48.651042,0.0,26.921823,1.213542,0.229167,14792.418652


**Análisis de resultados**

Los medoides iniciales se tomaron como el paciente más cercano al promedio de cada región, pero después de 3 iteraciones del K-Medoids los clusters dejaron de representar regiones y pasaron a agrupar pacientes por perfil de riesgo. Esto se ve en la distribución por región, donde cada región tiene pacientes en los 4 clusters.

Lo primero que destaca en los clusters es que se separan por sexo, edad y BMI: el Cluster 2 agrupa a todas las mujeres (662 pacientes, 39.5 años en promedio), el Cluster 0 a los hombres jóvenes (26 años), el Cluster 3 a hombres de 48.7 años con el menor BMI (26.92) y el Cluster 1 a hombres de 53 años con obesidad (BMI de 36.64).

El Cluster 1 tiene el mayor costo siniestro (19,289 USD) y solo el 2.5% de sus pacientes está en siniestralidad baja, mientras que el Cluster 0 tiene el menor costo (10,854 USD) con el 64.9% en siniestralidad baja. La edad y el BMI son las variables que más influyen en el costo.

Como US Health Insurance no quiere trasladar a sus pacientes, la recomendación es llevar los servicios a cada región según los clusters que más tiene. La región southeast tiene el mayor porcentaje de pacientes en el Cluster 1 (17.58%), por lo que ahí se deben priorizar programas de control de peso y prevención para adultos mayores. El Cluster 0, de bajo costo, puede atenderse con incentivos de prevención general.